# Análisis de la inversión en I+D en Argentina, 2004–2024

## Etapa 1 — Exploración, calidad de datos y análisis descriptivo

Este notebook analiza la evolución de la inversión en I+D en Argentina entre 2004 y 2024 a partir de cuatro hojas del archivo `inversion.xlsx`:

- **Recursos Financieros**
- **Sector**
- **Provincias**
- **Disciplinas**

El propósito de esta etapa es entender los datos, comprobar su consistencia y obtener resultados descriptivos que permitan responder:

> **¿Cómo evolucionó la inversión en I+D en Argentina entre 2004 y 2024 y cómo cambió su distribución entre sectores, territorios y disciplinas?**

El análisis se mantiene deliberadamente descriptivo. No se realizan modelos predictivos ni inferencias causales.

## Ficha general del proyecto

| Campo | Contenido |
|---|---|
| **Nombre del proyecto** | Análisis de la inversión en I+D en Argentina, 2004–2024 |
| **Especialidad** | Data Analytics (DA) |
| **Fuente del proyecto** | Dataset `inversion.xlsx` |
| **Fuente original** | **Pendiente de validar y documentar** |
| **Proyecto publicado** | **Pendiente de publicar en GitHub** |

> No se inventa el enlace de la fuente original. Antes de publicar, debe incorporarse el enlace oficial y comprobarse que corresponde al dataset utilizado.

## 1. Objetivo del proyecto

Analizar la evolución de la inversión en I+D en Argentina durante 2004–2024 y describir cómo se distribuye entre sectores ejecutores, provincias y disciplinas científicas.

El análisis busca generar evidencia descriptiva que pueda servir como punto de partida para investigadores, instituciones públicas u otros usuarios interesados en la evolución y distribución de los recursos destinados a I+D.

**Decisión que se busca apoyar:** identificar cambios relevantes en la magnitud y distribución de la inversión, así como posibles aspectos que requieran una revisión adicional de la fuente.

## 2. Plan de trabajo

1. **Exploración inicial:** estructura, variables, tipos de datos y categorías.
2. **Control de calidad:** valores faltantes, duplicados, claves y conciliación entre hojas.
3. **Análisis de Recursos Financieros:** evolución de las principales series e indicadores disponibles.
4. **Análisis de Sector, Provincias y Disciplinas:** distribución y cambios entre 2004 y 2024.
5. **Conclusiones y limitaciones:** principales hallazgos, aspectos pendientes y próximos pasos.

## 3. Carga de datos

Se utiliza el archivo local `../Data/raw/inversion.xlsx`. No se modifican los datos originales.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ruta_archivo = Path("../Data/raw/inversion.xlsx")

print("¿Existe el archivo?", ruta_archivo.exists())
print("Ruta:", ruta_archivo.resolve())

excel = pd.ExcelFile(ruta_archivo)

print("\nHojas disponibles:")
print(excel.sheet_names)

datos = {
    hoja: pd.read_excel(excel, sheet_name=hoja)
    for hoja in excel.sheet_names
}

print("\nHojas cargadas:", list(datos.keys()))

## 4. Exploración inicial

Primero se revisan dimensiones, columnas, tipos de datos y una muestra de cada hoja. Esta revisión permite confirmar la estructura antes de realizar transformaciones.

In [ ]:
for nombre, df in datos.items():
    print(f"\n{'=' * 60}")
    print(f"HOJA: {nombre}")
    print(f"{'=' * 60}")
    print("Dimensiones:", df.shape)
    print("Columnas:", df.columns.tolist())
    print("\nTipos de datos:")
    print(df.dtypes)
    print("\nPrimeras 3 filas:")
    display(df.head(3))

### Estructura encontrada

El archivo contiene cuatro hojas con información anual de 2004 a 2024:

- **Recursos Financieros:** 21 filas y 8 columnas.
- **Sector:** 105 filas y 3 columnas.
- **Provincias:** 504 filas y 3 columnas.
- **Disciplinas:** 147 filas y 3 columnas.

Las hojas desglosadas utilizan `ANIO` junto con una categoría: `SECT_EJEC`, `PROVINCIA` o `DISCIPL_ID`.

## 5. Calidad de datos

Se revisan tres controles básicos:

- valores faltantes;
- filas completamente duplicadas;
- unicidad de las combinaciones que identifican cada registro.

No se imputan ni eliminan datos en esta etapa.

In [ ]:
# Valores faltantes y duplicados
for nombre, df in datos.items():
    faltantes = df.isna().sum().sum()
    duplicados = df.duplicated().sum()
    print(f"{nombre}: {faltantes} valores faltantes | {duplicados} filas duplicadas")

In [ ]:
# Comprobación de claves esperadas
claves = {
    "Recursos Financieros": ["ANIO"],
    "Sector": ["ANIO", "SECT_EJEC"],
    "Provincias": ["ANIO", "PROVINCIA"],
    "Disciplinas": ["ANIO", "DISCIPL_ID"]
}

for nombre, columnas in claves.items():
    df = datos[nombre]
    duplicados_clave = df.duplicated(subset=columnas).sum()
    print(f"{nombre}: {duplicados_clave} claves duplicadas usando {columnas}")

### Resultado del control de calidad

No se detectaron valores faltantes ni filas completamente duplicadas en las cuatro hojas revisadas.

Las combinaciones `ANIO + categoría` funcionan como identificadores únicos en las hojas de Sector, Provincias y Disciplinas. No se realizaron correcciones porque no se encontraron registros que justificaran una imputación o eliminación.

## 6. Conciliación entre las hojas

Las hojas de Sector, Provincias y Disciplinas contienen desgloses que pueden compararse con `INV_ID_PESOS_CORR` de Recursos Financieros.

La conciliación sirve para detectar diferencias antes de interpretar los desgloses. Una diferencia no se corrige automáticamente: primero se documenta.

In [ ]:
recursos = datos["Recursos Financieros"].copy()
sector = datos["Sector"].copy()
provincias = datos["Provincias"].copy()
disciplinas = datos["Disciplinas"].copy()

total_recursos = (
    recursos
    .set_index("ANIO")["INV_ID_PESOS_CORR"]
    .sort_index()
)

# Totales anuales por hoja desglosada
totales_sector = sector.groupby("ANIO")["INV_ID_PESOS_CORR"].sum()
totales_provincias = provincias.groupby("ANIO")["INV_ID_PESOS_CORR"].sum()
totales_disciplinas = disciplinas.groupby("ANIO")["INV_ID_PESOS_CORR"].sum()

conciliacion = pd.DataFrame({
    "Total recursos": total_recursos,
    "Sector": totales_sector,
    "Provincias": totales_provincias,
    "Disciplinas": totales_disciplinas
})

conciliacion["Dif. Sector"] = conciliacion["Sector"] - conciliacion["Total recursos"]
conciliacion["Dif. Provincias"] = conciliacion["Provincias"] - conciliacion["Total recursos"]
conciliacion["Dif. Disciplinas"] = conciliacion["Disciplinas"] - conciliacion["Total recursos"]

display(conciliacion.round(2))

### Resultado de la conciliación

- **Provincias:** los totales anuales presentan diferencias muy pequeñas respecto de Recursos Financieros.
- **Sector:** la correspondencia es prácticamente exacta en la mayoría de los años, pero en **2023** el desglose sectorial suma 1,150,563 frente a 1,143,579 en Recursos Financieros, una diferencia de 6,984 (aprox. 0.61%).
- **Disciplinas:** la suma es inferior al total financiero durante el período. La causa de esta diferencia no puede determinarse únicamente con el archivo.

Por lo tanto, las diferencias se conservan como una **limitación documentada** y no se realizan ajustes para forzar la conciliación.

## 7. Análisis de Recursos Financieros

La hoja contiene varias medidas de inversión:

- pesos corrientes;
- pesos constantes;
- dólares corrientes;
- dólares PPC, según la denominación del archivo;
- inversión en I+D como porcentaje del PBI;
- participación pública y privada como porcentaje del PBI.

Primero se revisan las series monetarias mediante índices con base 2004 = 100. La interpretación es descriptiva y depende de las definiciones y unidades originales del archivo.

In [ ]:
recursos = recursos.sort_values("ANIO").reset_index(drop=True)

series_monetarias = [
    "INV_ID_PESOS_CORR",
    "INV_ID_PESOS_CONS",
    "INV_ID_DOL_CORR",
    "INV_ID_DOL_PPC"
]

indices = recursos[["ANIO"] + series_monetarias].copy()

for columna in series_monetarias:
    indices[columna] = (
        recursos[columna] / recursos.loc[0, columna] * 100
    ).round(2)

display(indices)

In [ ]:
# Visualización de las cuatro series indexadas
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

titulos = {
    "INV_ID_PESOS_CORR": "Pesos corrientes",
    "INV_ID_PESOS_CONS": "Pesos constantes",
    "INV_ID_DOL_CORR": "Dólares corrientes",
    "INV_ID_DOL_PPC": "Dólares PPC"
}

for ax, columna in zip(axes, series_monetarias):
    ax.plot(indices["ANIO"], indices[columna])
    ax.set_title(titulos[columna])
    ax.set_xlabel("Año")
    ax.set_ylabel("Índice (2004 = 100)")
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

### Principales observaciones de Recursos Financieros

- La serie de **pesos corrientes** presenta un crecimiento nominal muy pronunciado, especialmente entre 2023 y 2024. No debe interpretarse como crecimiento real porque está expresada en pesos corrientes.
- La serie de **pesos constantes según el archivo** aumenta hasta 2015, disminuye posteriormente, se recupera hasta 2023 y vuelve a caer en 2024.
- Los **dólares corrientes** muestran crecimiento hasta 2013, una caída en 2014, un máximo en 2015, retroceso posterior y recuperación hasta 2023, seguida de una caída en 2024.
- Los **dólares PPC**, manteniendo la denominación del archivo, muestran crecimiento hasta 2012, fluctuaciones posteriores, recuperación entre 2021 y 2023 y una caída en 2024.

Estas observaciones son descriptivas. Antes de utilizar las series para comparaciones económicas más profundas sería necesario validar las unidades, el año base y la metodología de cálculo de cada indicador.

# 8. Distribución por sector

La hoja Sector contiene cinco categorías. Para facilitar la interpretación se calcula la participación de cada sector sobre el total financiero anual.

Se utiliza el total de **Recursos Financieros** como denominador para mantener visible la diferencia detectada en 2023.

In [ ]:
sector_anual = sector.pivot(
    index="ANIO",
    columns="SECT_EJEC",
    values="INV_ID_PESOS_CORR"
)

sectores = [
    "Empresas",
    "Entidades privadas sin fines de lucro",
    "Organismos públicos de ciencia",
    "Universidades privadas",
    "Universidades públicas"
]

sector_anual = sector_anual[sectores]

participacion_sector = (
    sector_anual
    .div(total_recursos, axis=0)
    .mul(100)
    .round(2)
)

display(participacion_sector)

In [ ]:
cambio_participacion = (
    participacion_sector.loc[2024] -
    participacion_sector.loc[2004]
).round(2)

resultado_sector = pd.DataFrame({
    "Participación 2004 (%)": participacion_sector.loc[2004],
    "Participación 2024 (%)": participacion_sector.loc[2024],
    "Cambio (puntos porcentuales)": cambio_participacion
})

display(resultado_sector)

In [ ]:
comparacion_sector = participacion_sector.loc[[2004, 2024]].T

ax = comparacion_sector.plot(
    kind="bar",
    figsize=(10, 6)
)

ax.set_title("Participación sectorial de la inversión en I+D: 2004 vs. 2024")
ax.set_xlabel("Sector ejecutor")
ax.set_ylabel("Participación de la inversión total (%)")
plt.xticks(rotation=25, ha="right")
plt.legend(title="Año")
plt.tight_layout()
plt.show()

### Resultado del análisis sectorial

Entre 2004 y 2024, **Empresas** aumentó su participación de 32.98% a 47.52%, un incremento de **14.54 puntos porcentuales**.

En sentido contrario, **Universidades públicas** pasó de 22.97% a 14.45% (-8.52 puntos porcentuales) y **Organismos públicos de ciencia** de 39.61% a 35.04% (-4.57 puntos porcentuales).

Las entidades privadas sin fines de lucro y las universidades privadas tuvieron cambios menores.

Estos resultados describen un cambio en la **composición relativa** de la inversión. No permiten determinar las causas de esos cambios.

# 9. Distribución por provincias

La hoja Provincias contiene 24 categorías y 21 años.

Para mantener el análisis enfocado, se revisará:

1. la conciliación anual con Recursos Financieros;
2. la participación de cada provincia en 2004 y 2024;
3. las provincias con mayor inversión en 2024.

No se utilizarán 24 líneas simultáneas ni análisis de concentración avanzados, porque no son necesarios para responder el objetivo del proyecto.

In [ ]:
provincia_anual = provincias.pivot(
    index="ANIO",
    columns="PROVINCIA",
    values="INV_ID_PESOS_CORR"
)

participacion_provincias = (
    provincia_anual
    .div(total_recursos, axis=0)
    .mul(100)
    .round(2)
)

display(participacion_provincias.loc[[2004, 2024]])

In [ ]:
ranking_provincias_2024 = (
    provincias[provincias["ANIO"] == 2024]
    .sort_values("INV_ID_PESOS_CORR", ascending=False)
    .copy()
)

ranking_provincias_2024["Participación (%)"] = (
    ranking_provincias_2024["INV_ID_PESOS_CORR"]
    / total_recursos.loc[2024]
    * 100
).round(2)

display(ranking_provincias_2024.head(10))

In [ ]:
top_provincias = ranking_provincias_2024.head(10).sort_values(
    "INV_ID_PESOS_CORR"
)

ax = top_provincias.plot(
    x="PROVINCIA",
    y="Participación (%)",
    kind="barh",
    figsize=(9, 6),
    legend=False
)

ax.set_title("Principales provincias por participación en la inversión en I+D, 2024")
ax.set_xlabel("Participación de la inversión total (%)")
ax.set_ylabel("Provincia")
plt.tight_layout()
plt.show()

### Resultado del análisis provincial

La distribución territorial presenta diferencias importantes entre provincias. En 2024, el análisis permite identificar las provincias que concentran las mayores participaciones de la inversión registrada.

La lectura debe realizarse como **distribución de la inversión registrada por provincia**, sin atribuir causas a las diferencias territoriales.

La conciliación anual muestra además que la suma provincial es muy cercana al total de Recursos Financieros, por lo que este desglose presenta una consistencia interna alta dentro del archivo.

# 10. Distribución por disciplinas

La hoja Disciplinas contiene siete categorías. Se analizará su distribución en 2004 y 2024.

Debido a que la suma de esta hoja no coincide con el total de Recursos Financieros, sus porcentajes se calcularán respecto del **total registrado dentro de la propia hoja de Disciplinas**, no respecto del total financiero general. Así evitamos presentar las disciplinas como si representaran el 100% de la inversión total.

In [ ]:
disciplina_anual = disciplinas.pivot(
    index="ANIO",
    columns="DISCIPL_ID",
    values="INV_ID_PESOS_CORR"
)

participacion_disciplinas = (
    disciplina_anual
    .div(disciplina_anual.sum(axis=1), axis=0)
    .mul(100)
    .round(2)
)

display(participacion_disciplinas.loc[[2004, 2024]])

In [ ]:
comparacion_disciplinas = participacion_disciplinas.loc[[2004, 2024]].T

ax = comparacion_disciplinas.plot(
    kind="bar",
    figsize=(10, 6)
)

ax.set_title("Distribución de la inversión registrada por disciplina: 2004 vs. 2024")
ax.set_xlabel("Disciplina")
ax.set_ylabel("Participación dentro de la hoja Disciplinas (%)")
plt.xticks(rotation=25, ha="right")
plt.legend(title="Año")
plt.tight_layout()
plt.show()

### Resultado del análisis por disciplinas

La hoja permite observar cómo se distribuye la inversión que aparece registrada por disciplina y cómo cambia esa composición entre 2004 y 2024.

Sin embargo, esta distribución no debe interpretarse como la composición de la totalidad de la inversión en I+D, porque la suma de la hoja Disciplinas es inferior al total financiero. La causa de la diferencia no está explicada por el archivo y queda como una limitación del análisis.

# 11. Resumen de resultados

Los principales resultados descriptivos del análisis son:

- La inversión registrada muestra una evolución creciente en términos nominales, aunque las series monetarias requieren validar sus unidades y metodologías antes de realizar interpretaciones económicas más profundas.
- La composición sectorial cambió entre 2004 y 2024: las empresas aumentaron su participación y las universidades públicas redujeron la suya.
- La distribución provincial muestra diferencias importantes entre territorios, con una concentración visible en las provincias con mayor inversión registrada.
- La hoja de disciplinas permite analizar la composición de la inversión registrada por disciplina, pero no concilia con el total financiero y por ello debe interpretarse con cautela.
- Se detectó una diferencia puntual relevante entre Sector y Recursos Financieros en 2023, que no fue corregida ni ocultada.

# 12. Limitaciones

1. El archivo y sus variables todavía requieren validación documental de la fuente original, unidades y metodología.
2. `INV_ID_PESOS_CORR` representa una serie en pesos corrientes; sus variaciones no deben interpretarse como crecimiento real.
3. La diferencia de 2023 entre Sector y Recursos Financieros no tiene una causa determinada en este archivo.
4. La hoja Disciplinas no concilia con el total financiero y no se conoce la razón exacta de la diferencia.
5. El análisis es descriptivo y no permite establecer relaciones causales.

No se realizaron imputaciones, correcciones ni ajustes para forzar la conciliación.

# 13. Conclusiones y próximos pasos

El análisis permitió estructurar y revisar un conjunto de datos de inversión en I+D de 2004 a 2024, comprobar su calidad básica y describir su evolución y distribución.

El principal aprendizaje técnico fue trabajar con varias tablas relacionadas, validar su consistencia antes de analizarlas y distinguir entre valores absolutos y participación relativa.

Como mejora posterior, sería necesario validar la documentación oficial de la fuente, las definiciones de los indicadores y las diferencias observadas antes de realizar análisis económicos o de política pública más profundos.

Para la entrega actual, el análisis se mantiene dentro del alcance definido: exploración, preparación básica, análisis descriptivo, visualización, resultados y limitaciones.

# 14. Preguntas clave pendientes de validación

Antes de considerar definitivo el análisis, conviene confirmar con la fuente o el responsable del proyecto:

1. ¿Cuál es la fuente original y cuál es la metodología exacta de cada indicador financiero?
2. ¿Qué explica la diferencia entre la hoja Disciplinas y el total de Recursos Financieros?
3. ¿Qué explica específicamente la diferencia observada entre Sector y Recursos Financieros en 2023?

Estas preguntas se mantienen abiertas porque el notebook no debe inventar explicaciones que el dataset no permite demostrar.

# 15. Checklist previo a publicar

- [ ] README explica objetivo, fuente, metodología y principales resultados.
- [ ] Notebook ejecuta de principio a fin sin celdas de prueba o errores.
- [ ] No hay credenciales ni información sensible.
- [ ] La fuente original está documentada y el enlace fue validado.
- [ ] Los archivos están organizados.
- [ ] Las limitaciones y discrepancias están documentadas.
- [ ] El proyecto publicado en GitHub es accesible.